In [1]:
#====================================
#Celda 1 — Imports y configuración
#====================================
import json
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style="whitegrid")

DATA_PATH = Path("../../data/economico")
print(f"📁 Ruta de datos: {DATA_PATH.resolve()}")


📁 Ruta de datos: /workspaces/TFG_Spain-s_Migratory_Flow/data/economico


In [2]:
#=============================================
#Celda 2 — Cargar todos los JSONs disponibles
#=============================================

# Cargar todos los JSONs acumulados
archivos = sorted(glob.glob(str(DATA_PATH / "**/*.json"), recursive=True))
print(f"📦 JSONs encontrados: {len(archivos)}")
for f in archivos:
    print(f"  → {f}")

# Cargar el más reciente
with open(archivos[-1], "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"\n✅ JSON cargado: {archivos[-1]}")
print(f"🕐 Timestamp: {data.get('timestamp')}")
print(f"🔑 Claves disponibles: {list(data.keys())}")



📦 JSONs encontrados: 1
  → ../../data/economico/2026/04/13_04_2026.json



✅ JSON cargado: ../../data/economico/2026/04/13_04_2026.json
🕐 Timestamp: 2026-04-13T12:03:46.150103
🔑 Claves disponibles: ['timestamp', 'tasa_paro_actividad_provincia', 'ipv_precio_vivienda_ccaa', 'compraventas_vivienda', 'hipotecas_vivienda', 'renta_media_hogar', 'renta_media_hogar_provincia', 'renta_por_unidad_consumo', 'renta_neta_media_municipio', 'renta_neta_media_seccion', 'densidad_poblacion', 'turismo_viajeros', 'turismo_pernoctaciones', 'centros_educativos_ccaa', 'ipv_precio_vivienda_anual', 'crimen', 'centros_salud', 'catastro', 'alquiler_ministerio']


In [3]:
#============================================
# Celda 3 — Explorar tasa de paro por CCAA
#============================================
# La API INE devuelve lista de objetos con Nombre, MetaData y Data
paro_raw = data["tasa_paro_actividad_provincia"]   # ← clave actualizada
print(f"Tipo: {type(paro_raw)} | Registros: {len(paro_raw)}")
print(f"\nEstructura de un registro:")
print(json.dumps(paro_raw[0], indent=2, ensure_ascii=False))


Tipo: <class 'list'> | Registros: 477

Estructura de un registro:
{
  "COD": "EPA438090",
  "Nombre": "Tasa de actividad. Ambos sexos. Total Nacional. Total. ",
  "T3_Unidad": "Tasas",
  "T3_Escala": " ",
  "MetaData": [
    {
      "Id": 283908,
      "T3_Variable": "Tipo de dato",
      "Nombre": "Tasa de actividad",
      "Codigo": ""
    },
    {
      "Id": 454,
      "T3_Variable": "Sexo",
      "Nombre": "Ambos sexos",
      "Codigo": ""
    },
    {
      "Id": 16473,
      "T3_Variable": "Total Nacional",
      "Nombre": "Total Nacional",
      "Codigo": "00"
    },
    {
      "Id": 21387,
      "T3_Variable": "Nacionalidad",
      "Nombre": "Total",
      "Codigo": ""
    }
  ],
  "Data": [
    {
      "Fecha": "2025-10-01T00:00:00.000+02:00",
      "T3_TipoDato": "Definitivo",
      "T3_Periodo": "T4",
      "Anyo": 2025,
      "Valor": 58.94
    },
    {
      "Fecha": "2025-07-01T00:00:00.000+02:00",
      "T3_TipoDato": "Definitivo",
      "T3_Periodo": "T3",
      "Anyo

In [4]:
#============================================
# Celda 4 — Parsear tasa de paro a DataFrame
#============================================
def parse_ine_tabla(raw: list) -> pd.DataFrame:
    filas = []
    for serie in raw:
        metadata = serie.get("MetaData", [])
        datos    = serie.get("Data", [])
        dims = {}
        for m in metadata:
            col_name = m.get("Variable", m.get("T3_Variable", "dim")).strip().lower().replace(" ", "_")
            dims[col_name] = m.get("Nombre", "")
        for punto in datos:
            fila = {**dims}
            fila["valor"]   = punto.get("Valor")
            fila["fecha"]   = punto.get("Fecha") or punto.get("fecha")
            fila["anyo"]    = punto.get("Anyo")
            fila["secreto"] = punto.get("Secreto", False)
            if fila["fecha"]:
                mes = int(fila["fecha"][5:7])
                fila["trimestre"] = f"T{(mes - 1) // 3 + 1}"
            filas.append(fila)
    return pd.DataFrame(filas)

# Re-parsear con nombres reales
df_paro = parse_ine_tabla(data["tasa_paro_actividad_provincia"])   # ← clave actualizada
print(f"Shape: {df_paro.shape}")
print(f"Columnas: {list(df_paro.columns)}")
df_paro.head()


Shape: (45792, 10)
Columnas: ['tipo_de_dato', 'sexo', 'total_nacional', 'nacionalidad', 'valor', 'fecha', 'anyo', 'secreto', 'trimestre', 'provincias']


,tipo_de_dato,sexo,total_nacional,nacionalidad,valor,fecha,anyo,secreto,trimestre,provincias
0,Tasa de actividad,Ambos sexos,Total Nacional,Total,58.94,2025-10-01T00:00:00.000+02:00,2025,False,T4,NaN
1,Tasa de actividad,Ambos sexos,Total Nacional,Total,59.30,2025-07-01T00:00:00.000+02:00,2025,False,T3,NaN
2,Tasa de actividad,Ambos sexos,Total Nacional,Total,59.03,2025-04-01T00:00:00.000+02:00,2025,False,T2,NaN
3,Tasa de actividad,Ambos sexos,Total Nacional,Total,58.57,2025-01-01T00:00:00.000+01:00,2025,False,T1,NaN
4,Tasa de actividad,Ambos sexos,Total Nacional,Total,58.49,2024-10-01T00:00:00.000+02:00,2024,False,T4,NaN


In [5]:
#============================================
# Celda 5 — Análisis rápido de paro
#============================================
print("Columnas disponibles:")
print(list(df_paro.columns))

print("\nPrimeras fechas disponibles:")
print(df_paro["fecha"].sort_values(ascending=False).unique()[:10])

print("\nTrimestres disponibles:")
print(df_paro["trimestre"].unique())

print("\nAños disponibles:")
print(sorted(df_paro["anyo"].dropna().unique()))

print("\nValores nulos por columna:")
print(df_paro.isnull().sum())

print(f"\nRango de valores: {df_paro['valor'].min():.2f}% — {df_paro['valor'].max():.2f}%")
print(f"Media nacional: {df_paro['valor'].mean():.2f}%")


Columnas disponibles:
['tipo_de_dato', 'sexo', 'total_nacional', 'nacionalidad', 'valor', 'fecha', 'anyo', 'secreto', 'trimestre', 'provincias']

Primeras fechas disponibles:
<ArrowStringArray>
['2025-10-01T00:00:00.000+02:00', '2025-07-01T00:00:00.000+02:00',
 '2025-04-01T00:00:00.000+02:00', '2025-01-01T00:00:00.000+01:00',
 '2024-10-01T00:00:00.000+02:00', '2024-07-01T00:00:00.000+02:00',
 '2024-04-01T00:00:00.000+02:00', '2024-01-01T00:00:00.000+01:00',
 '2023-10-01T00:00:00.000+02:00', '2023-07-01T00:00:00.000+02:00']
Length: 10, dtype: str

Trimestres disponibles:
<ArrowStringArray>
['T4', 'T3', 'T2', 'T1']
Length: 4, dtype: str

Años disponibles:
[np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), n

In [6]:
#=============================================
#Celda 6 — Explorar todas las tablas INE
#=============================================
TABLAS = [
    "tasa_paro_ccaa",
    "renta_media_hogar",
    "densidad_poblacion",
    "turismo_viajeros",
    "turismo_pernoctaciones",
    "centros_educativos_ccaa",
    "ipv_precio_vivienda_ccaa",
    "ipv_precio_vivienda_anual",
    "compraventas_vivienda",
    "hipotecas_vivienda",
]

resumen = []
for tabla in TABLAS:
    raw = data.get(tabla, [])
    if isinstance(raw, list) and len(raw) > 0:
        df_tmp = parse_ine_tabla(raw)
        resumen.append({
            "tabla": tabla,
            "series": len(raw),
            "filas_total": len(df_tmp),
            "columnas": list(df_tmp.columns),
            "periodos": df_tmp["periodo"].nunique() if "periodo" in df_tmp.columns else 0,
            "nulos_%": round(df_tmp["valor"].isnull().mean() * 100, 1) if "valor" in df_tmp.columns else None
        })
    else:
        resumen.append({"tabla": tabla, "series": 0, "filas_total": 0, "error": str(raw)[:80]})

df_resumen = pd.DataFrame(resumen)
print("📊 Resumen de todas las tablas INE:")
df_resumen[["tabla", "series", "filas_total", "periodos", "nulos_%"]]


📊 Resumen de todas las tablas INE:


,tabla,series,filas_total,periodos,nulos_%
0,tasa_paro_ccaa,0,0,NaN,NaN
1,renta_media_hogar,32,576,0.00,0.70
2,densidad_poblacion,159,3975,0.00,0.00
3,turismo_viajeros,420,136910,0.00,0.10
4,turismo_pernoctaciones,441,131670,0.00,3.50
5,centros_educativos_ccaa,180,180,0.00,7.80
6,ipv_precio_vivienda_ccaa,240,17024,0.00,0.00
7,ipv_precio_vivienda_anual,120,2128,0.00,0.00
8,compraventas_vivienda,360,82440,0.00,0.00
9,hipotecas_vivienda,200,4460,0.00,0.00


In [7]:
#=============================================
#Celda 7 — Explorar Catastro
#=============================================
catastro = data.get("catastro", {})
for ciudad, info in catastro.items():
    print(f"\n🏗️  {ciudad.upper()}")
    print(f"  Status: {info.get('status')}")
    print(f"  Preview: {str(info.get('contenido_preview', ''))[:200]}")



🏗️  MADRID
  Status: 200
  Preview: <?xml version="1.0" encoding="utf-8"?>
<consulta_municipiero xmlns="http://www.catastro.meh.es/">
  <control>
    <cumun>4</cumun>
  </control>
  <municipiero>
    <muni>
      <nm>HUMANES DE M

🏗️  BARCELONA
  Status: 200
  Preview: <?xml version="1.0" encoding="utf-8"?>
<consulta_municipiero xmlns="http://www.catastro.meh.es/">
  <control>
    <cumun>1</cumun>
  </control>
  <municipiero>
    <muni>
      <nm>BARCELONA</n

🏗️  VALENCIA
  Status: 200
  Preview: <?xml version="1.0" encoding="utf-8"?>
<consulta_municipiero xmlns="http://www.catastro.meh.es/">
  <control>
    <cumun>1</cumun>
  </control>
  <municipiero>
    <muni>
      <nm>VALENCIA</nm

🏗️  SEVILLA
  Status: 200
  Preview: <?xml version="1.0" encoding="utf-8"?>
<consulta_municipiero xmlns="http://www.catastro.meh.es/">
  <control>
    <cumun>2</cumun>
  </control>
  <municipiero>
    <muni>
      <nm>EL CUERVO DE


In [8]:
#============================================
# Celda 8 — Visualización: Tasa de paro por CCAA (último periodo)
#============================================

# Ver fechas disponibles
print(f"Valores únicos en 'trimestre': {df_paro['trimestre'].unique()}")
print(f"Valores únicos en 'fecha' (muestra): {df_paro['fecha'].dropna().unique()[:5]}")

# Usar fecha como referencia temporal
df_paro["fecha_dt"] = pd.to_datetime(df_paro["fecha"], utc=True, errors="coerce")
ultima_fecha = df_paro["fecha_dt"].max()
print(f"\n📅 Última fecha disponible: {ultima_fecha}")

# Filtrar último periodo
df_paro_ultimo = df_paro[df_paro["fecha_dt"] == ultima_fecha].copy()
df_paro_ultimo = df_paro_ultimo.dropna(subset=["valor"]).sort_values("valor", ascending=True)

print(f"Registros en último periodo: {len(df_paro_ultimo)}")

# Ver columnas de dimensión disponibles
col_dims = [c for c in df_paro_ultimo.columns if c not in ["valor", "trimestre", "fecha", "fecha_dt", "secreto", "anyo"]]
print(f"Columnas dimensión: {col_dims}")
df_paro_ultimo[col_dims + ["valor"]].head(10)


Valores únicos en 'trimestre': <ArrowStringArray>
['T4', 'T3', 'T2', 'T1']
Length: 4, dtype: str
Valores únicos en 'fecha' (muestra): <ArrowStringArray>
['2025-10-01T00:00:00.000+02:00', '2025-07-01T00:00:00.000+02:00',
 '2025-04-01T00:00:00.000+02:00', '2025-01-01T00:00:00.000+01:00',
 '2024-10-01T00:00:00.000+02:00']
Length: 5, dtype: str

📅 Última fecha disponible: 2025-09-30 22:00:00+00:00
Registros en último periodo: 477
Columnas dimensión: ['tipo_de_dato', 'sexo', 'total_nacional', 'nacionalidad', 'provincias']


,tipo_de_dato,sexo,total_nacional,nacionalidad,provincias,valor
22752,Tasa de paro de la población,Hombres,NaN,Total,Huesca,2.55
22272,Tasa de paro de la población,Hombres,NaN,Total,Gipuzkoa,3.45
21408,Tasa de paro de la población,Hombres,NaN,Total,Burgos,3.55
17664,Tasa de paro de la población,Ambos sexos,NaN,Total,Huesca,4.69
23040,Tasa de paro de la población,Hombres,NaN,Total,Lleida,4.90
23712,Tasa de paro de la población,Hombres,NaN,Total,Palencia,5.60
17952,Tasa de paro de la población,Ambos sexos,NaN,Total,Lleida,5.73
24672,Tasa de paro de la población,Hombres,NaN,Total,Teruel,5.75
24960,Tasa de paro de la población,Hombres,NaN,Total,Valladolid,5.95
29568,Tasa de paro de la población,Mujeres,NaN,Total,Soria,6.17


In [9]:
#=============================================
#Celda 9 — Guardar DataFrames parseados para fase limpieza
#=============================================
output_path = Path("../../output/economico/01-raw")
output_path.mkdir(exist_ok=True)

# Guardar cada tabla como parquet para la fase de limpieza
for tabla in TABLAS:
    raw = data.get(tabla, [])
    if isinstance(raw, list) and len(raw) > 0:
        df_tmp = parse_ine_tabla(raw)
        df_tmp["fuente_tabla"] = tabla
        df_tmp["fecha_captura"] = data.get("timestamp")
        ruta = output_path / f"raw_{tabla}.parquet"
        df_tmp.to_parquet(ruta, index=False)
        print(f"✅ Guardado: {ruta} ({len(df_tmp)} filas)")

print("\n🎉 Todos los parquets guardados en output/")


✅ Guardado: ../../output/economico/01-raw/raw_renta_media_hogar.parquet (576 filas)
✅ Guardado: ../../output/economico/01-raw/raw_densidad_poblacion.parquet (3975 filas)


✅ Guardado: ../../output/economico/01-raw/raw_turismo_viajeros.parquet (136910 filas)


✅ Guardado: ../../output/economico/01-raw/raw_turismo_pernoctaciones.parquet (131670 filas)
✅ Guardado: ../../output/economico/01-raw/raw_centros_educativos_ccaa.parquet (180 filas)
✅ Guardado: ../../output/economico/01-raw/raw_ipv_precio_vivienda_ccaa.parquet (17024 filas)
✅ Guardado: ../../output/economico/01-raw/raw_ipv_precio_vivienda_anual.parquet (2128 filas)


✅ Guardado: ../../output/economico/01-raw/raw_compraventas_vivienda.parquet (82440 filas)
✅ Guardado: ../../output/economico/01-raw/raw_hipotecas_vivienda.parquet (4460 filas)

🎉 Todos los parquets guardados en output/


In [10]:
#=============================================
#
#=============================================
